<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula04c%20-%20cross%20validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [215]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [216]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6111111111111112


In [217]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6666666666666666


In [227]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)

best_acc = 0
best_k = 0

for k in range(1, 21, 2):
  model = KNeighborsClassifier(n_neighbors=k)
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  print(f"Accuracy for k={k}: {acc}")
  if acc > best_acc:
    best_acc = acc
    best_k = k

print(f"Best accuracy: {best_acc} for k={best_k}")

Accuracy for k=1: 0.6551724137931034
Accuracy for k=3: 0.6551724137931034
Accuracy for k=5: 0.6206896551724138
Accuracy for k=7: 0.5517241379310345
Accuracy for k=9: 0.7586206896551724
Accuracy for k=11: 0.6896551724137931
Accuracy for k=13: 0.6551724137931034
Accuracy for k=15: 0.7241379310344828
Accuracy for k=17: 0.6551724137931034
Accuracy for k=19: 0.6896551724137931
Best accuracy: 0.7586206896551724 for k=9


# cross validation

In [274]:
import numpy as np

def cross_validation(model, X, y, k=3):
  n = int(len(y)/k)
  idx = np.random.permutation(len(y))
  X = X[idx]
  y = y[idx]
  for i in range(k):
    X_tr = np.concatenate([X[:i*n], X[(i+1)*n:]])
    y_tr = np.concatenate([y[:i*n], y[(i+1)*n:]])
    X_val = X[i*n:(i+1)*n]
    y_val = y[i*n:(i+1)*n]
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    return acc

acc = cross_validation(model, X_train, y_train)
print(acc)

0.723404255319149


In [257]:
def search_best_k(model, X, y, k=3):
  best_acc = 0
  best_k = 0
  for k in range(1, 21, 2):
    model = KNeighborsClassifier(n_neighbors=k)
    acc = cross_validation(model, X, y)
    print(f"Accuracy for k={k}: {acc}")
    if acc > best_acc:
      best_acc = acc
      best_k = k
  print(f"Best accuracy: {best_acc} for k={best_k}")

search_best_k(model, X_train, y_train)

Accuracy for k=1: 0.723404255319149
Accuracy for k=3: 0.723404255319149
Accuracy for k=5: 0.6595744680851063
Accuracy for k=7: 0.7872340425531915
Accuracy for k=9: 0.574468085106383
Accuracy for k=11: 0.8723404255319149
Accuracy for k=13: 0.6382978723404256
Accuracy for k=15: 0.6170212765957447
Accuracy for k=17: 0.7021276595744681
Accuracy for k=19: 0.7659574468085106
Best accuracy: 0.8723404255319149 for k=11


In [281]:
def repeated_cross_validation(model, X, y, k=3, n=10):
  accs = []
  for i in range(n):
    acc = cross_validation(model, X, y)
    accs.append(acc)
  return np.mean(accs), np.std(accs)

mean, std = repeated_cross_validation(model, X_train, y_train)
print(mean, std)

0.7170212765957448 0.05713073013658531


In [296]:
def search_best_k(model, X, y, k=3):
  best_acc = 0
  best_k = 0
  for k in range(1, 21, 2):
    model = KNeighborsClassifier(n_neighbors=k)
    acc, _ = repeated_cross_validation(model, X, y)
    print(f"Accuracy for k={k}: {acc}")
    if acc > best_acc:
      best_acc = acc
      best_k = k
  print(f"Best accuracy: {best_acc} for k={best_k}")

search_best_k(model, X_train, y_train)

Accuracy for k=1: 0.7361702127659574
Accuracy for k=3: 0.7
Accuracy for k=5: 0.6702127659574468
Accuracy for k=7: 0.6659574468085105
Accuracy for k=9: 0.7127659574468084
Accuracy for k=11: 0.6978723404255319
Accuracy for k=13: 0.6914893617021277
Accuracy for k=15: 0.6978723404255319
Accuracy for k=17: 0.7170212765957447
Accuracy for k=19: 0.7255319148936169
Best accuracy: 0.7361702127659574 for k=1


In [305]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=3, shuffle=True)
accs = []
for train_index, test_index in kf.split(X_train):
  X_tr, X_val = X_train[train_index], X_train[test_index]
  y_tr, y_val = y_train[train_index], y_train[test_index]
  model = KNeighborsClassifier(n_neighbors=5)
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  accs.append(acc)

print(np.mean(accs))

0.7257683215130024


In [314]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=3, n_repeats=10)
accs = []
for train_index, test_index in rkf.split(X_train):
  X_tr, X_val = X_train[train_index], X_train[test_index]
  y_tr, y_val = y_train[train_index], y_train[test_index]
  model = KNeighborsClassifier(n_neighbors=5)
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  accs.append(acc)

print(np.mean(accs))

0.6761081560283688


In [318]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train, y_train,
                         scoring='accuracy',
                         cv=KFold(n_splits=3, shuffle=True))
print(scores)
print(np.mean(scores))

[0.66666667 0.76595745 0.68085106]
0.7044917257683215


In [319]:
scores = cross_val_score(model, X_train, y_train,
                         scoring='accuracy',
                         cv=RepeatedKFold(n_splits=3, n_repeats=10))
print(scores)
print(np.mean(scores))

[0.72916667 0.76595745 0.65957447 0.72916667 0.59574468 0.72340426
 0.625      0.78723404 0.78723404 0.75       0.78723404 0.59574468
 0.70833333 0.63829787 0.70212766 0.6875     0.72340426 0.78723404
 0.625      0.70212766 0.68085106 0.6875     0.72340426 0.65957447
 0.77083333 0.65957447 0.72340426 0.58333333 0.74468085 0.70212766]
0.70149231678487


In [320]:
def search_best_k(model, X, y, k=3):
  best_acc = 0
  best_k = 0
  for k in range(1, 21, 2):
    model = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(model, X_train, y_train,
                         scoring='accuracy',
                         cv=RepeatedKFold(n_splits=3, n_repeats=10))
    acc = np.mean(scores)
    print(f"Accuracy for k={k}: {acc}")
    if acc > best_acc:
      best_acc = acc
      best_k = k
  print(f"Best accuracy: {best_acc} for k={best_k}")

search_best_k(model, X_train, y_train)

Accuracy for k=1: 0.7336436170212767
Accuracy for k=3: 0.6908244680851064
Accuracy for k=5: 0.6808510638297872
Accuracy for k=7: 0.6979757683215129
Accuracy for k=9: 0.6788268321513002
Accuracy for k=11: 0.6998965721040187
Accuracy for k=13: 0.6831560283687943
Accuracy for k=15: 0.703619976359338
Accuracy for k=17: 0.7147901891252957
Accuracy for k=19: 0.7097369976359339
Best accuracy: 0.7336436170212767 for k=1


In [321]:
model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6111111111111112
